# MMGTFFF — Experiment 2 (Phase A): Hierarchical Tweet-Attention Encoder

### Embedding-agnostic tweet-level + day-level attention, ablated across 3 text backends, carried through the full FS1–FS4 grid.

Run top-to-bottom on a Colab **GPU** runtime (Runtime → Change runtime type → GPU).
Clones both `stocknet-dataset` (raw prices/tweets) and this project's own
`capstone` repo (for cached SEC EDGAR fundamentals — no new API calls needed).

## What this notebook answers

Your roadmap's Experiment 4 question: **does hierarchical tweet attention
outperform simple pooling, and does text-encoder choice matter once
aggregation is fixed?** Experiment 1 (exact StockNet reproduction) used a
flat tweet *count* as its only text signal (best result: LSTM on
Price+TweetCounts, MCC +0.092). This notebook replaces that count with an
actual **hierarchical attention encoder** over tweet content:

```
Company tweets for (ticker, day)
        |
per-tweet embedding (USE 512-d / FinBERT CLS 768-d / VADER 4-d)
        |
   projected to a shared 64-d space
        |
Tweet-level attention  (which tweets today mattered?)
        |
   one 64-d vector per day
        |
Day-level GRU + temporal attention  (which of the last 4 days mattered?)
        |
   one 64-d text embedding per sample
```

The encoder itself doesn't know or care which backend produced its input —
only the projection layer's input dimension changes. That's what makes the
3-way ablation (USE vs FinBERT vs VADER) a controlled comparison: the
architecture is held completely fixed.

## Scope (what this notebook is NOT)

- **No bilinear fusion, no GAT.** Price/fundamentals + the text embedding are
  combined by simple concatenation (late fusion). Bilinear fusion and the
  graph are Experiment 2's remaining pieces, still to come.
- **No event/global tweets.** Only company tweets are used here, same scope
  as Experiment 1's FS3.
- **Fundamentals are unchanged from Experiment 1** — raw 8 EDGAR features,
  zero-filled, not a learned embedding. (Building a real fundamentals
  encoder is a separate, smaller piece of Experiment 2 — this notebook is
  scoped to the text side only, since that's what's being ablated.)
- **LSTM/MLP/LR here are just classifier heads for a baseline comparison
  table**, not meaningful architecture choices in their own right — the
  actual interesting model is the tweet encoder feeding them.

## Design

All 3 embedding backends are carried through the **full FS1–FS4 grid** (not
just a standalone text classifier), per your instruction — so this notebook
runs 3 backends × 4 feature sets × 3 models = 36 evaluations, then reports
the winner. FS1/FS2 don't depend on the backend at all (no tweet feature),
so those 2×3=6 runs are identical across backends and are only computed
once.

## 1. Setup & Clone

In [ ]:
!git clone https://github.com/yumoxu/stocknet-dataset.git
!git clone https://github.com/AdityaMelkote3004/capstone.git
print('Cloned!')

In [ ]:
!pip install -q tensorflow-hub vaderSentiment
print('Installed.')

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from datetime import datetime, timedelta
from torch.utils.data import Dataset, DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

STOCKNET_ROOT = 'stocknet-dataset'
CAPSTONE_ROOT = 'capstone'
PRICE_DIR = os.path.join(STOCKNET_ROOT, 'price', 'preprocessed')
TWEET_DIR = os.path.join(STOCKNET_ROOT, 'tweet', 'preprocessed')

MAX_N_DAYS = 5
MAX_INPUT_DAYS = MAX_N_DAYS - 1   # 4
MAX_N_MSGS = 30                    # StockNet config.yml max_n_msgs

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if DEVICE.type != 'cuda':
    print('WARNING: no GPU detected -- USE/FinBERT embedding will be slow. '
          'Runtime -> Change runtime type -> GPU.')

## 2. Ticker Universe, Split Dates, Loaders

Identical to the Experiment 1 notebook — same 88 tickers, same official
split, same price/tweet loading.

In [ ]:
STOCK_SYMBOLS = (
    ['XOM','RDS-B','PTR','CVX','TOT','BP','BHP','SNP','SLB','BBL'] +
    ['AAPL','PG','BUD','KO','PM','TM','PEP','UN','UL','MO'] +
    ['JNJ','PFE','NVS','UNH','MRK','AMGN','MDT','ABBV','SNY','CELG'] +
    ['AMZN','BABA','WMT','CMCSA','HD','DIS','MCD','CHTR','UPS','PCLN'] +
    ['NEE','DUK','D','SO','NGG','AEP','PCG','EXC','SRE','PPL'] +
    ['IEP','HRG','CODI','REX','SPLP','PICO','AGFS','GMRE'] +
    ['BCH','BSAC','BRK-A','JPM','WFC','BAC','V','C','HSBC','MA'] +
    ['GE','MMM','BA','HON','UTX','LMT','CAT','GD','DHR','ABB'] +
    ['GOOG','MSFT','FB','T','CHL','ORCL','TSM','VZ','INTC','CSCO']
)
assert len(STOCK_SYMBOLS) == 88

SPLIT_DATES = {
    'train': ('2014-01-01', '2015-08-01'),
    'dev':   ('2015-08-01', '2015-10-01'),
    'test':  ('2015-10-01', '2016-01-01'),
}

def get_split(date):
    ds = date.isoformat()
    for name, (start, end) in SPLIT_DATES.items():
        if start <= ds < end:
            return name
    return None


def load_movement_file(ticker):
    fp = os.path.join(PRICE_DIR, f'{ticker}.txt')
    if not os.path.exists(fp):
        return None
    rows = []
    with open(fp, 'r', encoding='utf8') as f:
        for line in f:
            parts = line.rstrip('\n').split('\t')
            date = datetime.strptime(parts[0], '%Y-%m-%d').date()
            mv = float(parts[1])
            prices = [float(parts[3]), float(parts[4]), float(parts[5])]
            rows.append((date, mv, prices))
    rows.sort(key=lambda r: r[0])
    return rows


def load_tweet_dates(ticker):
    tdir = os.path.join(TWEET_DIR, ticker)
    out = {}
    if not os.path.isdir(tdir):
        return out
    for fname in os.listdir(tdir):
        try:
            d = datetime.strptime(fname, '%Y-%m-%d').date()
        except ValueError:
            continue
        msgs = []
        try:
            with open(os.path.join(tdir, fname), 'r', encoding='utf8') as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    obj = json.loads(line)
                    text = obj.get('text', '')
                    if isinstance(text, list) and text:
                        msgs.append(text)
        except Exception:
            continue
        if msgs:
            out[d] = msgs
    return out

print('Loaders defined.')

## 3. Build Samples — Same Windowing as Experiment 1, Plus Tweet References

Same buffer-zone filter, calendar-anchored window, and trading-day tweet
alignment as Experiment 1. The difference: instead of collapsing each
aligned day's tweets to a scalar count, we keep a `(date, msg_index)`
reference for every attached tweet, so embeddings can be computed **once per
unique tweet** (see Section 5) rather than redundantly once per sample.

In [ ]:
def build_samples_with_tweet_refs(ticker, movement_rows, tweet_by_date, max_n_days=MAX_N_DAYS):
    date_list = [r[0] for r in movement_rows]
    mv_by_date = {r[0]: r[1] for r in movement_rows}
    prices_by_date = {r[0]: r[2] for r in movement_rows}
    date_set = set(date_list)
    max_input_days = max_n_days - 1

    samples = []
    for main_target_date in date_list:
        main_mv = mv_by_date[main_target_date]
        if -0.005 <= main_mv < 0.0055:
            continue

        d_t_min = main_target_date - timedelta(days=max_n_days - 1)
        ts_window = sorted(d for d in date_set if d_t_min <= d < main_target_date)
        T_input = len(ts_window)
        if T_input == 0:
            continue

        ts_full = ts_window + [main_target_date]

        d_d_max = main_target_date - timedelta(days=1)
        d_d_min = main_target_date - timedelta(days=max_n_days)
        unaligned_days = sorted(d for d in tweet_by_date if d_d_min <= d <= d_d_max)

        # slot_refs[t] = list of (date, msg_index) tuples attached to slot t
        slot_refs = [[] for _ in range(T_input + 1)]
        for d in unaligned_days:
            n_msgs = len(tweet_by_date[d])
            for t in range(T_input + 1):
                if d < ts_full[t]:
                    slot_refs[t].extend((d, i) for i in range(n_msgs))
                    break

        input_prices = [prices_by_date[d] for d in ts_window]
        input_prices += [[0.0, 0.0, 0.0]] * (max_input_days - T_input)

        input_tweet_refs = [refs[:MAX_N_MSGS] for refs in slot_refs[:-1]]
        input_tweet_refs += [[]] * (max_input_days - T_input)

        samples.append({
            'Ticker': ticker,
            'Target_Date': main_target_date,
            'Target': 1 if main_mv > 0 else 0,
            'Window_Length': T_input,
            'Input_Prices': input_prices,
            'Input_Tweet_Refs': input_tweet_refs,  # (max_input_days) lists of (date, msg_idx)
        })

    return samples


all_samples = []
tweet_by_date_all = {}  # ticker -> {date: [token_list, ...]}
for ticker in STOCK_SYMBOLS:
    movement_rows = load_movement_file(ticker)
    if not movement_rows:
        continue
    tweet_by_date = load_tweet_dates(ticker)
    tweet_by_date_all[ticker] = tweet_by_date
    samples = build_samples_with_tweet_refs(ticker, movement_rows, tweet_by_date)
    for s in samples:
        s['Split'] = get_split(s['Target_Date'])
    all_samples.extend(s for s in samples if s['Split'] is not None)

print(f'Total samples: {len(all_samples)}')
assert 25000 < len(all_samples) < 28000, \
    f'Sample count {len(all_samples)} unexpected -- check windowing logic against Experiment 1.'
print('Sample count matches the expected StockNet dataset scale.')

## 4. Fundamentals (FS2/FS4) — Reused, Unchanged, From the Cloned Repo

Same as Experiment 1: forward-filled from SEC filing date, aligned against
each ticker's **full** trading calendar, zero-filled where unavailable. Uses
the cached raw EDGAR fetch already in the repo — **no new API calls**.

In [ ]:
fund_raw = pd.read_csv(os.path.join(CAPSTONE_ROOT, 'dataset', 'final', 'edgar_raw_fundamentals.csv'))
fund_raw['Filed_Date'] = pd.to_datetime(fund_raw['Filed_Date']).dt.date
FUNDAMENTAL_FEATURES = ['Revenue', 'NetIncome', 'TotalAssets', 'TotalLiabilities',
                         'StockholdersEquity', 'EPS', 'Cash']
N_FUND = len(FUNDAMENTAL_FEATURES) + 1  # +ROA

def align_fundamentals_for_ticker(ticker, all_dates):
    daily_index = pd.DatetimeIndex(pd.to_datetime(all_dates))
    ticker_data = fund_raw[fund_raw['Ticker'] == ticker]
    result = {}
    if len(ticker_data) == 0:
        for m in FUNDAMENTAL_FEATURES:
            result[m] = pd.Series(np.nan, index=daily_index)
        result['ROA'] = pd.Series(np.nan, index=daily_index)
        return result
    for metric in FUNDAMENTAL_FEATURES:
        md_ = (ticker_data[ticker_data['Metric'] == metric]
               .sort_values('Filed_Date').drop_duplicates(subset=['Filed_Date'], keep='last'))
        if len(md_) == 0:
            result[metric] = pd.Series(np.nan, index=daily_index)
            continue
        ts = md_.set_index(pd.to_datetime(md_['Filed_Date']))['Value']
        ts = ts[~ts.index.duplicated(keep='last')]
        result[metric] = ts.reindex(daily_index, method='ffill')
    roa = result['NetIncome'] / result['TotalAssets']
    result['ROA'] = roa.replace([np.inf, -np.inf], np.nan)
    return result


fund_lookup_by_ticker = {}
for ticker in STOCK_SYMBOLS:
    movement_rows = load_movement_file(ticker)
    if not movement_rows:
        continue
    all_dates = [r[0] for r in movement_rows]
    fund_series = align_fundamentals_for_ticker(ticker, all_dates)
    fund_lookup_by_ticker[ticker] = {
        d: [fund_series[m].iloc[i] for m in FUNDAMENTAL_FEATURES + ['ROA']]
        for i, d in enumerate(all_dates)
    }

for s in all_samples:
    ticker, target = s['Ticker'], s['Target_Date']
    lookup = fund_lookup_by_ticker.get(ticker, {})
    d_t_min = target - timedelta(days=MAX_N_DAYS - 1)
    ts_window = sorted(d for d in lookup if d_t_min <= d < target)
    fund_window = [lookup[d] for d in ts_window]
    fund_window += [[np.nan] * N_FUND] * (MAX_INPUT_DAYS - len(ts_window))
    s['Input_Fundamentals'] = fund_window

print('Fundamentals aligned.')

## 5. Per-Tweet Embedding Functions

**Fix for the OOM crash**: an earlier version of this notebook computed all 3
backends' embeddings up front and attached a dense
`(4 days x 30 tweets x embedding_dim)` tensor for **all 3 backends
simultaneously** to every one of the 26,619 samples. Sizing that:
`26,619 x 4 x 30 x 768 x 4 bytes ~= 9.3GB` for FinBERT alone, another ~6GB for
USE, on top of the model weights themselves — comfortably exceeding Colab's
RAM on any tier.

The fix: each backend's model is loaded, used to embed every **unique**
tweet exactly once into one flat `(N, dim)` array, then the model is deleted
before the next backend runs — at most one backend's model is ever in memory
at a time. The big per-sample `(4, 30, dim)` window tensors are never
materialized for the whole dataset at all; they're built lazily, one batch
at a time, inside the `Dataset.__getitem__` below (`TweetRefDataset`), from
each sample's lightweight `(date, msg_index)` references plus this flat
embeddings array via a lookup dict. A batch of 64 samples costs
`64 x 4 x 30 x 768 x 4 bytes ~= 23MB` — trivial, regardless of dataset size.

In [ ]:
import gc

def flatten_all_tweets(tweet_by_date_all):
    flat_keys, flat_texts = [], []
    for ticker, by_date in tweet_by_date_all.items():
        for date, msgs in by_date.items():
            for i, tokens in enumerate(msgs):
                flat_keys.append((ticker, date, i))
                flat_texts.append(' '.join(tokens))
    return flat_keys, flat_texts


flat_keys, flat_texts = flatten_all_tweets(tweet_by_date_all)
key_to_row_global = {k: i for i, k in enumerate(flat_keys)}
print(f'Unique tweets to embed: {len(flat_texts)}')


def embed_use(texts, batch_size=512):
    import tensorflow as tf
    import tensorflow_hub as hub
    model = hub.load('https://tfhub.dev/google/universal-sentence-encoder/4')
    out = np.zeros((len(texts), 512), dtype=np.float32)
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        out[start:start + len(batch)] = model(batch).numpy()
        if start % (batch_size * 20) == 0:
            print(f'  USE: {start}/{len(texts)}')
    del model
    tf.keras.backend.clear_session()
    gc.collect()
    return out


def embed_finbert(texts, batch_size=64, max_length=64):
    from transformers import AutoTokenizer, AutoModel
    tok = AutoTokenizer.from_pretrained('ProsusAI/finbert')
    model = AutoModel.from_pretrained('ProsusAI/finbert').to(DEVICE).eval()
    out = np.zeros((len(texts), 768), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, len(texts), batch_size):
            batch = texts[start:start + batch_size]
            inputs = tok(batch, padding=True, truncation=True, max_length=max_length, return_tensors='pt')
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
            hidden = model(**inputs).last_hidden_state[:, 0, :]
            out[start:start + len(batch)] = hidden.cpu().numpy()
            if start % (batch_size * 50) == 0:
                print(f'  FinBERT: {start}/{len(texts)}')
    del model, tok
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    gc.collect()
    return out


def embed_vader(texts):
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    analyzer = SentimentIntensityAnalyzer()
    out = np.zeros((len(texts), 4), dtype=np.float32)
    for i, text in enumerate(texts):
        s = analyzer.polarity_scores(text)
        out[i] = [s['neg'], s['neu'], s['pos'], s['compound']]
    return out


BACKEND_CONFIGS = {
    'USE':     (embed_use, 512),
    'FinBERT': (embed_finbert, 768),
    'VADER':   (embed_vader, 4),
}
print('Embedding functions defined (not yet run -- run one at a time in Section 7).')

## 6. The Embedding-Agnostic Hierarchical Attention Encoder

Same module regardless of backend — only the input projection's dimension
changes. Masking guarantees days with zero tweets contribute a zero vector,
never NaN from an all-masked softmax.

In [ ]:
class TweetLevelAttention(nn.Module):
    def __init__(self, input_dim, proj_dim=64):
        super().__init__()
        self.proj = nn.Linear(input_dim, proj_dim)
        self.score = nn.Linear(proj_dim, 1)

    def forward(self, tweet_embs, tweet_mask):
        # tweet_embs: (B, D, M, E), tweet_mask: (B, D, M)
        h = torch.tanh(self.proj(tweet_embs))
        scores = self.score(h).squeeze(-1)
        scores = scores.masked_fill(tweet_mask == 0, -1e9)
        has_any = (tweet_mask.sum(dim=-1, keepdim=True) > 0).float()
        weights = torch.softmax(scores, dim=-1) * has_any
        daily = (weights.unsqueeze(-1) * h).sum(dim=2)  # (B, D, P)
        return daily


class HierarchicalTweetEncoder(nn.Module):
    def __init__(self, input_dim, proj_dim=64, hidden_dim=64):
        super().__init__()
        self.tweet_attn = TweetLevelAttention(input_dim, proj_dim)
        self.gru = nn.GRU(proj_dim, hidden_dim, batch_first=True)
        self.temporal_score = nn.Linear(hidden_dim, 1)
        self.output_dim = hidden_dim

    def forward(self, tweet_embs, tweet_mask, window_mask):
        daily = self.tweet_attn(tweet_embs, tweet_mask)
        lengths = window_mask.sum(dim=1).clamp(min=1).long().cpu()
        packed = nn.utils.rnn.pack_padded_sequence(daily, lengths, batch_first=True, enforce_sorted=False)
        gru_out, _ = self.gru(packed)
        gru_out, _ = nn.utils.rnn.pad_packed_sequence(gru_out, batch_first=True, total_length=daily.shape[1])
        scores = self.temporal_score(gru_out).squeeze(-1)
        scores = scores.masked_fill(window_mask == 0, -1e9)
        weights = torch.softmax(scores, dim=-1)
        text_emb = (weights.unsqueeze(-1) * gru_out).sum(dim=1)
        return text_emb


def compute_metrics(y_true, y_pred, y_prob):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'mcc': matthews_corrcoef(y_true, y_pred),
        'auc': roc_auc_score(y_true, y_prob) if len(set(y_true)) > 1 else 0.5,
    }

print('Encoder defined.')

## 7. Per-Backend: Embed -> Train Stage A -> Extract Frozen Embeddings -> Free Memory

`TweetRefDataset` builds each sample's `(4, 30, dim)` window tensor lazily,
per batch, from the flat embeddings array -- this is the memory fix described
in Section 5. Each backend is processed fully (embed all tweets, train the
standalone Stage A classifier, extract the frozen 64-d embedding for every
sample) before its large embeddings array and model are explicitly deleted
and garbage-collected, so at most one backend's raw per-tweet embeddings are
ever resident in RAM.

In [ ]:
class TweetRefDataset(Dataset):
    """Builds the (days, msgs, dim) window tensor lazily per __getitem__ call
    from a flat (N, dim) embeddings array + (ticker,date,idx)->row lookup,
    instead of materializing dense per-sample tensors for the whole dataset
    up front (that's what caused the earlier OOM crash)."""
    def __init__(self, samples, embeddings, key_to_row, emb_dim):
        self.samples = samples
        self.embeddings = embeddings
        self.key_to_row = key_to_row
        self.emb_dim = emb_dim

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        s = self.samples[i]
        te = np.zeros((MAX_INPUT_DAYS, MAX_N_MSGS, self.emb_dim), dtype=np.float32)
        tm = np.zeros((MAX_INPUT_DAYS, MAX_N_MSGS), dtype=np.float32)
        ticker = s['Ticker']
        for day_i, refs in enumerate(s['Input_Tweet_Refs']):
            for msg_i, (date, idx) in enumerate(refs[:MAX_N_MSGS]):
                row = self.key_to_row.get((ticker, date, idx))
                if row is not None:
                    te[day_i, msg_i] = self.embeddings[row]
                    tm[day_i, msg_i] = 1.0
        wmask = np.zeros(MAX_INPUT_DAYS, dtype=np.float32)
        wmask[:s['Window_Length']] = 1.0
        return (torch.tensor(te), torch.tensor(tm), torch.tensor(wmask),
                torch.tensor(s['Target'], dtype=torch.long))


class TextClassifier(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.fc = nn.Linear(encoder.output_dim, 2)
    def forward(self, te, tm, wm):
        return self.fc(self.encoder(te, tm, wm))


def train_text_encoder(emb_dim, embeddings, key_to_row, train_samples, val_samples, test_samples,
                        epochs=30, patience=8):
    torch.manual_seed(SEED)
    encoder = HierarchicalTweetEncoder(input_dim=emb_dim).to(DEVICE)
    model = TextClassifier(encoder).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss()

    train_dl = DataLoader(TweetRefDataset(train_samples, embeddings, key_to_row, emb_dim),
                           batch_size=64, shuffle=True)
    val_ds = TweetRefDataset(val_samples, embeddings, key_to_row, emb_dim)
    test_ds = TweetRefDataset(test_samples, embeddings, key_to_row, emb_dim)

    def evaluate(ds):
        model.eval()
        dl = DataLoader(ds, batch_size=256)
        yt, yp, ypr = [], [], []
        with torch.no_grad():
            for te, tm, wm, y in dl:
                te, tm, wm = te.to(DEVICE), tm.to(DEVICE), wm.to(DEVICE)
                logits = model(te, tm, wm)
                probs = torch.softmax(logits, dim=1)[:, 1]
                yt.extend(y.numpy()); yp.extend(logits.argmax(1).cpu().numpy()); ypr.extend(probs.cpu().numpy())
        return compute_metrics(np.array(yt), np.array(yp), np.array(ypr))

    best_mcc, best_state, wait = -1e9, None, 0
    for epoch in range(1, epochs + 1):
        model.train()
        for te, tm, wm, y in train_dl:
            te, tm, wm, y = te.to(DEVICE), tm.to(DEVICE), wm.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(te, tm, wm), y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        val_m = evaluate(val_ds)
        if val_m['mcc'] > best_mcc:
            best_mcc, best_state, wait = val_m['mcc'], {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            wait += 1
        if wait >= patience:
            break
    model.load_state_dict(best_state)
    return encoder, evaluate(test_ds)


@torch.no_grad()
def extract_text_embeddings(samples, encoder, embeddings, key_to_row, emb_dim):
    encoder.eval()
    dl = DataLoader(TweetRefDataset(samples, embeddings, key_to_row, emb_dim), batch_size=256, shuffle=False)
    out = []
    for te, tm, wm, _ in dl:
        te, tm, wm = te.to(DEVICE), tm.to(DEVICE), wm.to(DEVICE)
        out.append(encoder(te, tm, wm).cpu().numpy())
    return np.concatenate(out, axis=0)


by_split = {'train': [], 'dev': [], 'test': []}
for s in all_samples:
    by_split[s['Split']].append(s)

# Cache each backend's raw per-tweet embeddings to disk. If the runtime
# crashes and restarts, re-running this cell reloads a finished backend's
# embeddings from disk instead of recomputing them from scratch.
CACHE_DIR = 'final/tweet_embeddings_cache'
os.makedirs(CACHE_DIR, exist_ok=True)

stage_a_results = {}
for backend_name, (embed_fn, emb_dim) in BACKEND_CONFIGS.items():
    cache_path = os.path.join(CACHE_DIR, f'{backend_name}_embeddings.npy')
    if os.path.exists(cache_path):
        print(f'{backend_name}: loading cached embeddings from {cache_path}')
        embeddings = np.load(cache_path)
    else:
        print(f"\n{'='*50}\n{backend_name}: embedding {len(flat_texts)} tweets...\n{'='*50}")
        embeddings = embed_fn(flat_texts)  # (N, emb_dim) -- only ONE backend's raw embeddings in memory at a time
        np.save(cache_path, embeddings)
        print(f'{backend_name}: cached embeddings to {cache_path} '
              f'(re-running this cell after a crash/restart will load from disk instead of recomputing)')

    print(f'{backend_name}: training standalone text encoder (Stage A)...')
    encoder, test_m = train_text_encoder(
        emb_dim, embeddings, key_to_row_global, by_split['train'], by_split['dev'], by_split['test'])
    stage_a_results[backend_name] = test_m
    print(f"  Test: Acc={test_m['accuracy']:.4f} F1={test_m['f1']:.4f} MCC={test_m['mcc']:+.4f} AUC={test_m['auc']:.4f}")

    print(f'{backend_name}: extracting frozen 64-d embeddings for the FS1-4 grid...')
    for split_name in ['train', 'dev', 'test']:
        embs = extract_text_embeddings(by_split[split_name], encoder, embeddings, key_to_row_global, emb_dim)
        for s, e in zip(by_split[split_name], embs):
            s[f'TextEmbedding_{backend_name}'] = e  # tiny (64,) vector -- cheap to keep

    # Free this backend's raw per-tweet embeddings (the big array) before the next backend runs.
    del embeddings, encoder
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    gc.collect()
    print(f'{backend_name}: done, memory freed.')

print('\nStage A summary (standalone text-only classifier):')
for b, m in stage_a_results.items():
    print(f"  {b:8s}: MCC={m['mcc']:+.4f} F1={m['f1']:.4f} AUC={m['auc']:.4f}")

## 8. Stage B — FS1–FS4 Grid × 3 Backends × 3 Models

Same official split, same window/mask convention, same model trio as
Experiment 1. FS3/FS4 use the frozen text embedding from Section 9 as a
**concatenated** feature (late fusion, not bilinear — that's Experiment 2's
remaining piece).

- LR / MLP: flatten the window and concatenate the 64-d text embedding.
- LSTM: run price [+fundamentals] through the masked LSTM, concatenate the
  final hidden state with the 64-d text embedding, then classify.

In [ ]:
def to_xy(samples, use_fund, backend):
    n = len(samples)
    n_feat = 3 + (N_FUND if use_fund else 0)
    X = np.zeros((n, MAX_INPUT_DAYS, n_feat), dtype=np.float32)
    mask = np.zeros((n, MAX_INPUT_DAYS), dtype=np.float32)
    text_emb = np.zeros((n, 64), dtype=np.float32) if backend else None
    y = np.zeros(n, dtype=np.int64)
    for i, s in enumerate(samples):
        prices = np.array(s['Input_Prices'], dtype=np.float32)
        X[i, :, :3] = prices
        if use_fund:
            fund = np.nan_to_num(np.array(s['Input_Fundamentals'], dtype=np.float32), nan=0.0)
            X[i, :, 3:3+N_FUND] = fund
        if backend:
            text_emb[i] = s[f'TextEmbedding_{backend}']
        mask[i, :s['Window_Length']] = 1.0
        y[i] = s['Target']
    return X, y, mask, text_emb


def normalize_window(X_train, *others):
    flat = X_train.reshape(-1, X_train.shape[-1])
    mean, std = flat.mean(axis=0), flat.std(axis=0)
    std[std == 0] = 1.0
    return [(X_train - mean) / std] + [(X - mean) / std for X in others]


def normalize_flat(X_train, *others):
    mean, std = X_train.mean(axis=0), X_train.std(axis=0)
    std[std == 0] = 1.0
    return [(X_train - mean) / std] + [(X - mean) / std for X in others]


class GridDataset(Dataset):
    def __init__(self, X, y, mask, text_emb):
        self.X, self.y, self.mask = X, y, mask
        self.text_emb = text_emb if text_emb is not None else np.zeros((len(y), 0), dtype=np.float32)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return (torch.tensor(self.X[idx]), torch.tensor(self.mask[idx]),
                torch.tensor(self.text_emb[idx]), torch.tensor(self.y[idx]))


class LSTMWithLateFusion(nn.Module):
    def __init__(self, input_dim, text_dim, hidden_dim=64):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=1, batch_first=True)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim + text_dim, 32), nn.ReLU(), nn.Dropout(0.2), nn.Linear(32, 2))

    def forward(self, x, mask, text_emb):
        lengths = mask.sum(dim=1).clamp(min=1).long().cpu()
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        _, (h_n, _) = self.lstm(packed)
        combined = torch.cat([h_n[-1], text_emb], dim=-1)
        return self.head(combined)


class MLPBaseline(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 2))
    def forward(self, x_flat):
        return self.net(x_flat)


def run_lstm(X_train, y_train, m_train, te_train, X_val, y_val, m_val, te_val,
             X_test, y_test, m_test, te_test, input_dim, text_dim, epochs=50, patience=10):
    torch.manual_seed(SEED)
    model = LSTMWithLateFusion(input_dim, text_dim).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss()
    train_dl = DataLoader(GridDataset(X_train, y_train, m_train, te_train), batch_size=64, shuffle=True)
    val_ds = GridDataset(X_val, y_val, m_val, te_val)
    test_ds = GridDataset(X_test, y_test, m_test, te_test)

    def evaluate(ds):
        model.eval()
        dl = DataLoader(ds, batch_size=256)
        yt, yp, ypr = [], [], []
        with torch.no_grad():
            for x, m, te, y in dl:
                x, m, te = x.to(DEVICE), m.to(DEVICE), te.to(DEVICE)
                logits = model(x, m, te)
                probs = torch.softmax(logits, dim=1)[:, 1]
                yt.extend(y.numpy()); yp.extend(logits.argmax(1).cpu().numpy()); ypr.extend(probs.cpu().numpy())
        return compute_metrics(np.array(yt), np.array(yp), np.array(ypr))

    best_mcc, best_state, wait = -1e9, None, 0
    for epoch in range(1, epochs + 1):
        model.train()
        for x, m, te, y in train_dl:
            x, m, te, y = x.to(DEVICE), m.to(DEVICE), te.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(x, m, te), y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        val_m = evaluate(val_ds)
        if val_m['mcc'] > best_mcc:
            best_mcc, best_state, wait = val_m['mcc'], {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            wait += 1
        if wait >= patience:
            break
    model.load_state_dict(best_state)
    return evaluate(test_ds)


def run_mlp(X_train, y_train, te_train, X_val, y_val, te_val, X_test, y_test, te_test,
            epochs=50, patience=10):
    torch.manual_seed(SEED)
    Xf_train = np.concatenate([X_train.reshape(len(X_train), -1), te_train], axis=1)
    Xf_val   = np.concatenate([X_val.reshape(len(X_val), -1), te_val], axis=1)
    Xf_test  = np.concatenate([X_test.reshape(len(X_test), -1), te_test], axis=1)
    model = MLPBaseline(Xf_train.shape[-1]).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss()
    train_dl = DataLoader(list(zip(torch.tensor(Xf_train), torch.tensor(y_train))), batch_size=64, shuffle=True)

    def evaluate(Xf, y):
        model.eval()
        with torch.no_grad():
            logits = model(torch.tensor(Xf).to(DEVICE))
            probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
            preds = logits.argmax(1).cpu().numpy()
        return compute_metrics(y, preds, probs)

    best_mcc, best_state, wait = -1e9, None, 0
    for epoch in range(1, epochs + 1):
        model.train()
        for xb, yb in train_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        val_m = evaluate(Xf_val, y_val)
        if val_m['mcc'] > best_mcc:
            best_mcc, best_state, wait = val_m['mcc'], {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            wait += 1
        if wait >= patience:
            break
    model.load_state_dict(best_state)
    return evaluate(Xf_test, y_test)


def run_logreg(X_train, y_train, te_train, X_test, y_test, te_test):
    Xf_train = np.concatenate([X_train.reshape(len(X_train), -1), te_train], axis=1)
    Xf_test = np.concatenate([X_test.reshape(len(X_test), -1), te_test], axis=1)
    model = LogisticRegression(max_iter=1000, random_state=SEED)
    model.fit(Xf_train, y_train)
    pred = model.predict(Xf_test)
    prob = model.predict_proba(Xf_test)[:, 1]
    return compute_metrics(y_test, pred, prob)


print('Stage B harness defined.')

In [ ]:
FEATURE_SETS = {
    'FS1_Price':                          (False, None),
    'FS2_Price_Fundamentals':             (True,  None),
    'FS3_Price_HierAttnTweets':           (False, 'BACKEND'),
    'FS4_Price_Fundamentals_HierAttnTweets': (True, 'BACKEND'),
}

all_results = {}  # all_results[fs_key or f'{fs_key}__{backend}'] = {...}

# FS1 / FS2 do not depend on backend -- compute once
for fs_key, (use_fund, _) in [('FS1_Price', (False, None)), ('FS2_Price_Fundamentals', (True, None))]:
    print(f"\n{'='*60}\n{fs_key}\n{'='*60}")
    X_train, y_train, m_train, _ = to_xy(by_split['train'], use_fund, None)
    X_val, y_val, m_val, _       = to_xy(by_split['dev'], use_fund, None)
    X_test, y_test, m_test, _    = to_xy(by_split['test'], use_fund, None)
    X_train, X_val, X_test = normalize_window(X_train, X_val, X_test)
    zeros = lambda n: np.zeros((n, 0), dtype=np.float32)
    te_train, te_val, te_test = zeros(len(y_train)), zeros(len(y_val)), zeros(len(y_test))
    input_dim = X_train.shape[-1]

    lr_m = run_logreg(X_train, y_train, te_train, X_test, y_test, te_test)
    lstm_m = run_lstm(X_train, y_train, m_train, te_train, X_val, y_val, m_val, te_val,
                       X_test, y_test, m_test, te_test, input_dim, 0)
    mlp_m = run_mlp(X_train, y_train, te_train, X_val, y_val, te_val, X_test, y_test, te_test)
    print(f"  LR:   MCC={lr_m['mcc']:+.4f} F1={lr_m['f1']:.4f}")
    print(f"  LSTM: MCC={lstm_m['mcc']:+.4f} F1={lstm_m['f1']:.4f}")
    print(f"  MLP:  MCC={mlp_m['mcc']:+.4f} F1={mlp_m['f1']:.4f}")
    all_results[fs_key] = {
        'label': fs_key, 'backend': None, 'num_features': int(input_dim),
        'models': {'Logistic Regression': {**lr_m, 'n_samples': len(y_test)},
                   'LSTM': {**lstm_m, 'n_samples': len(y_test)},
                   'MLP': {**mlp_m, 'n_samples': len(y_test)}}
    }

# FS3 / FS4 depend on backend -- run for all 3
for backend_name in BACKEND_CONFIGS:
    for fs_key, use_fund in [('FS3_Price_HierAttnTweets', False),
                              ('FS4_Price_Fundamentals_HierAttnTweets', True)]:
        result_key = f'{fs_key}__{backend_name}'
        print(f"\n{'='*60}\n{fs_key} [{backend_name}]\n{'='*60}")
        X_train, y_train, m_train, te_train = to_xy(by_split['train'], use_fund, backend_name)
        X_val, y_val, m_val, te_val         = to_xy(by_split['dev'], use_fund, backend_name)
        X_test, y_test, m_test, te_test     = to_xy(by_split['test'], use_fund, backend_name)
        X_train, X_val, X_test = normalize_window(X_train, X_val, X_test)
        te_train, te_val, te_test = normalize_flat(te_train, te_val, te_test)
        input_dim = X_train.shape[-1]

        lr_m = run_logreg(X_train, y_train, te_train, X_test, y_test, te_test)
        lstm_m = run_lstm(X_train, y_train, m_train, te_train, X_val, y_val, m_val, te_val,
                           X_test, y_test, m_test, te_test, input_dim, 64)
        mlp_m = run_mlp(X_train, y_train, te_train, X_val, y_val, te_val, X_test, y_test, te_test)
        print(f"  LR:   MCC={lr_m['mcc']:+.4f} F1={lr_m['f1']:.4f}")
        print(f"  LSTM: MCC={lstm_m['mcc']:+.4f} F1={lstm_m['f1']:.4f}")
        print(f"  MLP:  MCC={mlp_m['mcc']:+.4f} F1={mlp_m['f1']:.4f}")
        all_results[result_key] = {
            'label': fs_key, 'backend': backend_name, 'num_features': int(input_dim),
            'models': {'Logistic Regression': {**lr_m, 'n_samples': len(y_test)},
                       'LSTM': {**lstm_m, 'n_samples': len(y_test)},
                       'MLP': {**mlp_m, 'n_samples': len(y_test)}}
        }

print('\nAll 36 (well, 6+30) runs complete.')

## 9. Summary Table, Winner Selection, and Save

In [ ]:
rows = []
for key, res in all_results.items():
    for model_name, m in res['models'].items():
        rows.append({
            'FeatureSet': res['label'], 'Backend': res['backend'] or '-', 'Model': model_name,
            'Accuracy': m['accuracy'], 'F1': m['f1'], 'MCC': m['mcc'], 'AUC': m['auc'],
        })
results_df = pd.DataFrame(rows).sort_values('MCC', ascending=False).reset_index(drop=True)
print(results_df.to_string(index=False))

best_row = results_df.iloc[0]
print(f"\nBest overall: {best_row['FeatureSet']} [{best_row['Backend']}] / {best_row['Model']} "
      f"-- MCC={best_row['MCC']:+.4f}, F1={best_row['F1']:.4f}")

print('\nStage A standalone text-classifier comparison (embedding-choice ablation):')
for b, m in stage_a_results.items():
    print(f"  {b:8s}: MCC={m['mcc']:+.4f} F1={m['f1']:.4f} AUC={m['auc']:.4f}")

os.makedirs('final', exist_ok=True)
with open('final/hier_attn_stage_a_results.json', 'w') as f:
    json.dump(stage_a_results, f, indent=2)
with open('final/hier_attn_stage_b_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)
results_df.to_csv('final/hier_attn_summary.csv', index=False)
print('\nSaved: final/hier_attn_stage_a_results.json, final/hier_attn_stage_b_results.json, final/hier_attn_summary.csv')

## 10. Compare Against Experiment 1

| Feature Set | Experiment 1 (best) | This notebook (best) |
|---|---:|---:|
| FS1: Price | MLP, MCC +0.048 | *(see Section 11 table)* |
| FS2: Price + Fundamentals | LogReg, MCC +0.069 | *(see Section 11 table)* |
| FS3: Price + Tweets | **LSTM, MCC +0.092** (raw tweet count) | *(see Section 11 table — hierarchical attention)* |
| FS4: Full | LogReg, MCC +0.067 | *(see Section 11 table)* |

If FS3/FS4 here beat Experiment 1's tweet-count numbers, that's evidence
hierarchical attention over actual tweet content adds real signal beyond a
flat count — the answer to this notebook's core question. Whichever backend
wins is the one to carry into the full MAN-SF reproduction (bilinear fusion
+ GAT) as Experiment 2's remaining pieces.